In [1]:
import os
from traccuracy import EdgeFlag
from traccuracy.loaders import load_geff_data, load_tiffs
from traccuracy.matchers import CTCMatcher
from traccuracy.metrics import CTCMetrics

res_zarr = '/home/ddon0001/PhD/experiments/scaled/pre-thesis/scaled_w_merge/Fluo-N2DL-HeLa_01/matched_solution.zarr'
gt_seg_path = '/home/ddon0001/PhD/data/cell_tracking_challenge/SUBMISSION/Fluo-N2DL-HeLa/01_GT/TRA'
pred_seg_path = '/home/ddon0001/PhD/data/cell_tracking_challenge/SUBMISSION/Fluo-N2DL-HeLa/01_ERR_SEG'

gt_geff = os.path.join(res_zarr, 'gt.geff')
pred_geff = os.path.join(res_zarr, 'pred.geff')

gt_g = load_geff_data(gt_geff, load_all_props=True, seg_property='segmentation_id')
pred_g = load_geff_data(pred_geff, load_all_props=True, seg_property='label')

gt_seg = load_tiffs(gt_seg_path)
pred_seg = load_tiffs(pred_seg_path)

gt_g.segmentation = gt_seg
gt_g.label_key = 'segmentation_id'
pred_g.segmentation = pred_seg
pred_g.label_key = 'label'
pred_g.graph.remove_edges_from(list(pred_g.graph.edges()))
pred_g.edges_by_flag[EdgeFlag.CTC_FALSE_POS] = set()
pred_g.edges_by_flag[EdgeFlag.WRONG_SEMANTIC] = set()

Loading TIFFs: 100%|██████████| 91/91 [00:00<00:00, 131.43it/s]


In [2]:
matched = CTCMatcher().compute_mapping(gt_g, pred_g)
results = CTCMetrics().compute(matched)
print(results.results)

Matching frames:   0%|          | 0/92 [00:00<?, ?it/s]/home/ddon0001/PhD/code/traccuracy/src/traccuracy/matchers/_ctc.py:94: UserWarning: 'res_boxes' and/or 'res_labels' are not provided, using 'regionprops' to get them
  overlaps = get_labels_with_overlap(
Evaluating nodes: 100%|██████████| 8602/8602 [00:00<00:00, 322344.75it/s]
Evaluating FP edges: 0it [00:00, ?it/s]
Evaluating FN edges: 100%|██████████| 8562/8562 [00:00<00:00, 344390.18it/s]

{'AOGM': 13213.0, 'fp_nodes': 0, 'fn_nodes': 37, 'ns_nodes': 0, 'fp_edges': 0, 'fn_edges': 8562, 'ws_edges': 0, 'TRA': 0.866848729757238, 'DET': 0.9957170968862137, 'LNK': 0.0}


In [16]:
from traccuracy import EdgeFlag
[edge for edge in pred_g.graph.edges() if EdgeFlag.CTC_FALSE_POS in pred_g.graph.edges[edge]]

[]